# Lab 2 - Agentic Design Patterns + Bulk Email Mail Merge

**Contributor:** Akhila Guska ([@akhilaguska27](https://github.com/akhilaguska27))
**Course:** AI Engineer Agentic Track - The Complete Agent & MCP Course

This notebook covers the Lab 2 exercise:
1. Identifies the agentic design patterns used in the lab
2. Explains what 1 line changed this from a workflow to an agent
3. Adds a bulk email mail merge tool to send to multiple prospects

---

## Exercise Part 1: Agentic Design Patterns Used

The lab demonstrated these agentic design patterns:

**1. Parallelization**
All 3 sales agents ran at the same time using asyncio.gather(). This is faster than running them one by one.

**2. Tool use**
The @function_tool decorator converted a plain Python function into a tool the agent can call.
No JSON boilerplate needed - the SDK handles it automatically.

**3. Agent as tool**
sales_agent1.as_tool() converted an entire agent into a tool another agent can call.
This means agents can use other agents as building blocks.

**4. Orchestrator pattern**
The Sales Manager directed all 3 agents, evaluated results, and made decisions.
One agent controlling others is the orchestrator pattern.

**5. Handoff**
The Sales Manager passed control to the Email Manager to handle formatting and sending.
With handoffs, control passes permanently - unlike tools where control returns.

---

## Exercise Part 2: What 1 Line Changed Workflow to Agent?

```python
handoffs=[emailer_agent]
```

Without this line, the Sales Manager followed a fixed hardcoded path - that is a workflow.
With this line, the Sales Manager decides on its own when and whether to hand off - that is an agent.

This matches Anthropic's definition: a system where the LLM controls the workflow, not the programmer.

---

## Exercise Part 3: Mail Merge - Send to Multiple Prospects

The extension below adds a bulk email tool so the winning email gets sent to a list of prospects,
not just one recipient.

In [ ]:
# Imports - same as the original lab plus Dict for type hints
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [ ]:
# Load API keys from .env file
# Make sure SENDGRID_API_KEY and OPENAI_API_KEY are set
load_dotenv(override=True)

In [ ]:
# SSL fix for Mac users - run this if you get SSL certificate errors
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()

In [ ]:
# Three sales agents with different writing styles
# Each will write a different version of the cold email

instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-4o-mini")
sales_agent2 = Agent(name="Engaging Sales Agent", instructions=instructions2, model="gpt-4o-mini")
sales_agent3 = Agent(name="Busy Sales Agent", instructions=instructions3, model="gpt-4o-mini")

In [ ]:
# Convert each sales agent into a tool the Sales Manager can call
# This is the "Agent as tool" pattern
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3]

In [ ]:
# Subject writer and HTML converter agents
# These handle formatting before sending

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [ ]:
# MAIL MERGE TOOL - the main addition in this contribution
# Instead of sending to one person, this sends to a list of prospects
# Each email is personalized with the prospect's name
#
# To use this for real outreach:
# - Replace the emails in the prospects list with real recipient emails
# - Keep your verified sender email in from_email
# - The agent will automatically personalize each email

@function_tool
def send_bulk_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send the winning email to a list of sales prospects via mail merge.
    Each email is personalized with the prospect name."""

    # Define your list of prospects here
    # In a real system this could be loaded from a CSV or database
    prospects = [
        {"name": "CEO at TechCorp", "email": "your-email@gmail.com"},
        {"name": "CTO at StartupXYZ", "email": "your-email@gmail.com"},
        {"name": "VP Engineering at BigCo", "email": "your-email@gmail.com"},
    ]

    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    sent_count = 0

    for prospect in prospects:
        # Personalize the email body for each prospect
        personalized_body = html_body.replace("[CEO's Name]", prospect["name"])

        from_email = Email("your-verified-sender@gmail.com")  # Change to your verified sender
        to_email = To(prospect["email"])
        content = Content("text/html", personalized_body)
        mail = Mail(from_email, to_email, subject, content).get()
        sg.client.mail.send.post(request_body=mail)
        sent_count += 1
        print(f"Sent to: {prospect['name']}")

    return {"status": "success", "emails_sent": sent_count}

In [ ]:
# Email Manager agent - handles formatting and bulk sending
# This agent receives the winning email and:
# 1. Writes a compelling subject line
# 2. Converts the body to HTML
# 3. Sends to all prospects via mail merge

emailer_instructions = """You are an email formatter and bulk sender. 
You receive the body of an email to be sent to multiple prospects.
Follow these steps:
1. Use subject_writer to write a compelling subject line
2. Use html_converter to convert the body to HTML
3. Use send_bulk_email to send to all prospects
Only send once - do not call send_bulk_email more than once."""

emailer_agent = Agent(
    name="Email Manager",
    instructions=emailer_instructions,
    tools=[subject_tool, html_tool, send_bulk_email],
    model="gpt-4o-mini",
    handoff_description="Format the email and send to all prospects via mail merge"
)

In [ ]:
# Sales Manager agent - the orchestrator
# This is the top-level agent that:
# 1. Gets 3 email drafts from the sales agents (parallelization happens inside)
# 2. Picks the best one
# 3. Hands off to Email Manager for bulk sending
#
# The handoffs=[emailer_agent] line is what makes this an AGENT not just a workflow
# Without it, the path would be hardcoded - with it, the LLM decides when to hand off

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email
and send it to a list of prospects.

Follow these steps:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts.
   Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email.

3. Handoff for Sending: Pass ONLY the winning email draft to the Email Manager agent.
   The Email Manager will handle formatting and sending to all prospects.

Crucial Rules:
- Use the sales agent tools to generate drafts - do not write them yourself.
- Hand off exactly ONE email to the Email Manager - never more than one.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],  # This 1 line makes it an agent not a workflow
    model="gpt-4o-mini"
)

In [ ]:
# Run the full multi-agent pipeline
# Watch the output to see each agent working
# Then check your email inbox for the bulk emails

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Bulk email SDR"):
    result = await Runner.run(sales_manager, message)

print("Pipeline complete! Check your inbox for the bulk emails.")
print("Also check the trace at: https://platform.openai.com/traces")

---

## What This Pipeline Does

```
Sales Manager (orchestrator)
        |
        |-- calls sales_agent1 tool  --> professional email draft
        |-- calls sales_agent2 tool  --> humorous email draft
        |-- calls sales_agent3 tool  --> concise email draft
        |
        |-- picks the best draft
        |
        handoff --> Email Manager
                        |
                        |-- subject_writer tool  --> compelling subject line
                        |-- html_converter tool  --> HTML formatted body
                        |-- send_bulk_email tool --> sent to all prospects
```

## Commercial Applications

This pattern is directly applicable to:
- Sales automation at any company doing cold outreach
- HR teams sending personalized job offer emails
- Marketing teams running personalized campaigns
- Any business process that needs AI-generated personalized communications at scale

The same orchestrator + handoff pattern works for any multi-step business process,
not just email. You could replace the email tools with Slack messages, CRM updates,
calendar invites, or any other action.

---
*Contributed by Akhila Guska - feel free to build on this!*